# 🌽 Corn Yield Prediction — End-to-End ML Pipeline (v2)

**Predict:** `Wet weight in field (Kg m⁻²)`

**Inputs:** Seed variety, Cob Height (cm), Plant Height (cm), Cob Wet Weight (g), Cob Length (cm), Number of Seed Rows

---
### 📌 Key Concepts for Beginners

**Why 3 data splits?**
| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 70% | The model *learns* from this data |
| **Validation** | 15% | We *tune* the model using this — it's like a practice exam |
| **Test** | 15% | Final *unseen* evaluation — the real exam. Touched only once! |

> ⚠️ If you only use Train + Test, you risk overfitting your choices to the test set without realising it.

**What is "Accuracy" in regression?**
Regression models don't give a % accuracy like classifiers. Instead we use:
- **R²** — How much variance the model explains. 1.0 = perfect, 0 = no better than guessing the mean
- **MAE** — Average error in Kg m⁻² (easy to interpret)
- **RMSE** — Like MAE but punishes big errors more
- **MAPE** — Error as a percentage of the actual value

**Why encode Seed Variety?**
ML models only understand *numbers*, not text like `'Jet 999'`. We use **One-Hot Encoding (OHE)** which converts each variety into a column of 0s and 1s. This is the correct approach for regression — explained in detail in Section 4.

## 1 — Install & Import

In [ ]:
!pip install xgboost optuna shap --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import optuna
import shap
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('✅ All packages imported successfully')

## 2 — Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # Upload: maize_training_dataset_real_fields.xlsx

In [ ]:
df = pd.read_excel(
    'maize_training_dataset_real_fields.xlsx',
    sheet_name='ML_Training_Data'
)
df.columns = [
    'seed_variety', 'cob_height_cm', 'plant_height_cm',
    'cob_wet_weight_g', 'cob_length_cm', 'num_seed_rows', 'wet_weight_field'
]
print(f'Dataset shape: {df.shape}')
print(f'Seed varieties: {df["seed_variety"].unique()}')
df.head()

## 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Missing Values ==='); print(df.isnull().sum())
print('\n=== Descriptive Stats ==='); print(df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['seed_variety'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Sample Count per Seed Variety'); axes[0].tick_params(axis='x', rotation=30)
df['wet_weight_field'].plot(kind='hist', bins=20, ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Target Distribution — Wet Weight in Field')
axes[1].set_xlabel('Wet Weight (Kg m⁻²)')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix'); plt.tight_layout(); plt.show()

In [ ]:
# Average yield per seed variety — confirms variety is a major factor
variety_avg = df.groupby('seed_variety')['wet_weight_field'].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 4))
variety_avg.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Average Wet Weight by Seed Variety (confirms variety matters a lot!)')
plt.ylabel('Avg Wet Weight (Kg m⁻²)'); plt.xlabel('Seed Variety')
plt.xticks(rotation=25); plt.tight_layout(); plt.show()
print(variety_avg)

## 4 — Categorical Encoding: Seed Variety → Numbers

### Why we must encode it
ML models can only work with numbers. `'Jet 999'` means nothing to them. We need to convert seed variety into numbers.

### Why One-Hot Encoding (OHE) — NOT Label Encoding
You might think: just assign numbers — `Jet 999 = 1`, `GT 709 = 2`, etc. That's called **Label Encoding** and it's *wrong for regression* because it implies `GT 709` is "twice" `Jet 999`, which is meaningless.

**One-Hot Encoding** instead creates a separate column for each variety:

| seed_variety | is_GT200 | is_GT709 | is_Pacific808 | is_Jet999 | (Commando = all zeros, the reference) |
|---|---|---|---|---|---|
| Jet 999   | 0 | 0 | 0 | 1 | |
| GT 200    | 1 | 0 | 0 | 0 | |
| Commando  | 0 | 0 | 0 | 0 | ← `drop='first'` makes one variety the baseline |

> ✅ This is the **correct and standard approach** for regression with nominal categorical data. You are right that seed variety is a major factor — OHE ensures the model treats each variety independently.

In [ ]:
# Preview what OHE does to seed_variety
demo_ohe = OneHotEncoder(drop='first', sparse_output=False)
demo_encoded = demo_ohe.fit_transform(df[['seed_variety']])
demo_cols = demo_ohe.get_feature_names_out(['seed_variety'])
demo_df = pd.DataFrame(demo_encoded, columns=demo_cols)
demo_df.insert(0, 'seed_variety', df['seed_variety'].values)
print('One-Hot Encoded Seed Variety (first 10 rows):')
print(demo_df.drop_duplicates('seed_variety').to_string(index=False))
print('\n→ Each variety becomes its own 0/1 column. The model can now learn')
print('  a separate weight for each variety independently.')

## 5 — Feature Engineering & 3-Way Data Split

### The split strategy
```
Full Dataset (175 samples)
        │
        ├── Train      70%  (122 samples)  ← model learns here
        ├── Validation 15%  (26 samples)   ← we tune/select models here
        └── Test       15%  (27 samples)   ← final honest evaluation
```

In [ ]:
# --- Feature Engineering ---
df['cob_to_plant_ratio'] = df['cob_height_cm'] / df['plant_height_cm']   # ear position
df['weight_per_row']     = df['cob_wet_weight_g'] / df['num_seed_rows']   # seed density

FEATURES = [
    'seed_variety',
    'cob_height_cm', 'plant_height_cm',
    'cob_wet_weight_g', 'cob_length_cm',
    'num_seed_rows',
    'cob_to_plant_ratio', 'weight_per_row'
]
TARGET   = 'wet_weight_field'
CAT_COLS = ['seed_variety']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

X = df[FEATURES].copy()
y = df[TARGET].copy()

# --- 3-Way Split ---
# Step 1: Split off test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
# Step 2: Split remaining into train (70%) and validation (15%)
# 0.15 / 0.85 ≈ 0.176 gives us 15% of total for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42
)

total = len(X)
print(f'Total samples   : {total}')
print(f'Train  set      : {len(X_train)} samples ({len(X_train)/total*100:.0f}%)')
print(f'Validation set  : {len(X_val)} samples ({len(X_val)/total*100:.0f}%)')
print(f'Test set        : {len(X_test)} samples ({len(X_test)/total*100:.0f}%)')

# --- Shared Preprocessor ---
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CAT_COLS)
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)

## 6 — Helper Functions

In [ ]:
def evaluate_split(name, model_or_pipe, X_tr, y_tr, X_v, y_v, X_te, y_te, use_pipe=True):
    """
    Evaluate a model on Train, Validation, and Test sets.
    Returns a DataFrame with all metrics for all 3 splits.
    """
    def metrics(y_true, y_pred, split_name):
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        return {'Split': split_name, 'R²': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}

    predict = model_or_pipe.predict
    rows = [
        metrics(y_tr, predict(X_tr), 'Train'),
        metrics(y_v,  predict(X_v),  'Validation'),
        metrics(y_te, predict(X_te), 'Test')
    ]
    result_df = pd.DataFrame(rows).set_index('Split')

    print(f'\n{"="*52}')
    print(f'  {name} — Accuracy on All 3 Splits')
    print(f'{"="*52}')
    print(result_df.round(4).to_string())
    print(f'\n  💡 Interpretation:')
    gap = result_df.loc['Train', 'R²'] - result_df.loc['Test', 'R²']
    if gap > 0.15:
        print(f'     R² gap (Train-Test) = {gap:.3f} → possible OVERFITTING')
    elif gap < 0:
        print(f'     R² gap (Train-Test) = {gap:.3f} → Test better than Train (unusual, check data)')
    else:
        print(f'     R² gap (Train-Test) = {gap:.3f} → Good generalisation ✅')

    return result_df


def plot_predictions(y_true, y_pred, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    lim = [min(y_true.min(), y_pred.min()) - 0.05,
           max(y_true.max(), y_pred.max()) + 0.05]
    axes[0].scatter(y_true, y_pred, alpha=0.7, edgecolors='k', s=60)
    axes[0].plot(lim, lim, 'r--', lw=2, label='Perfect fit')
    axes[0].set(xlim=lim, ylim=lim, xlabel='Actual (Kg m⁻²)',
                ylabel='Predicted (Kg m⁻²)', title=f'{title} — Actual vs Predicted (Test Set)')
    axes[0].legend()
    residuals = np.array(y_true) - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.7, edgecolors='k', s=60)
    axes[1].axhline(0, color='red', linestyle='--', lw=2)
    axes[1].set(xlabel='Predicted (Kg m⁻²)', ylabel='Residual',
                title=f'{title} — Residual Plot (Test Set)')
    plt.tight_layout(); plt.show()


all_results = {}   # stores result DataFrames for final comparison
print('✅ Helper functions ready')

---
## Model 1 — Baseline: Ridge Regression
> Linear model with L2 regularisation. Fast and interpretable — sets the performance floor.

In [ ]:
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

# Cross-validation on training set
cv_r2   = cross_val_score(ridge_pipe, X_train, y_train, cv=kf, scoring='r2')
cv_rmse = -cross_val_score(ridge_pipe, X_train, y_train, cv=kf,
                            scoring='neg_root_mean_squared_error')
print(f'5-Fold CV R²  : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'5-Fold CV RMSE: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}')

# Train model
ridge_pipe.fit(X_train, y_train)

# Evaluate on ALL 3 splits
all_results['Ridge (Baseline)'] = evaluate_split(
    'Ridge Regression (Baseline)',
    ridge_pipe, X_train, y_train, X_val, y_val, X_test, y_test
)

# Actual vs Predicted on test set
plot_predictions(y_test.values, ridge_pipe.predict(X_test), 'Ridge Regression')

In [ ]:
# Feature coefficients
cat_names = list(ridge_pipe.named_steps['preprocessor']
                  .named_transformers_['cat'].get_feature_names_out(CAT_COLS))
feature_names = NUM_COLS + cat_names

coef_df = pd.DataFrame({'Feature': feature_names,
                         'Coefficient': ridge_pipe.named_steps['model'].coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(9, 5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='RdBu_r')
plt.axvline(0, color='black', lw=0.8)
plt.title('Ridge Regression — Feature Coefficients\n(larger absolute value = more influence)')
plt.tight_layout(); plt.show()

---
## Model 2 — More Accurate: Random Forest Regressor
> Ensemble of 300 decision trees — handles non-linear relationships and feature interactions.

In [ ]:
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300, max_depth=None,
        min_samples_split=4, min_samples_leaf=2,
        max_features='sqrt', random_state=42, n_jobs=-1
    ))
])

cv_r2_rf   = cross_val_score(rf_pipe, X_train, y_train, cv=kf, scoring='r2')
cv_rmse_rf = -cross_val_score(rf_pipe, X_train, y_train, cv=kf,
                               scoring='neg_root_mean_squared_error')
print(f'5-Fold CV R²  : {cv_r2_rf.mean():.4f} ± {cv_r2_rf.std():.4f}')
print(f'5-Fold CV RMSE: {cv_rmse_rf.mean():.4f} ± {cv_rmse_rf.std():.4f}')

rf_pipe.fit(X_train, y_train)

all_results['Random Forest (Accurate)'] = evaluate_split(
    'Random Forest (Accurate)',
    rf_pipe, X_train, y_train, X_val, y_val, X_test, y_test
)

plot_predictions(y_test.values, rf_pipe.predict(X_test), 'Random Forest')

In [ ]:
fi_df = pd.DataFrame({'Feature': feature_names,
                       'Importance': rf_pipe.named_steps['model'].feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
plt.title('Random Forest — Feature Importances\n(higher = more important to predictions)')
plt.tight_layout(); plt.show()

---
## Model 3 — Best: XGBoost + Optuna Hyperparameter Tuning
> Gradient boosted trees with 100-trial Bayesian hyperparameter search.
> Optuna uses the **validation set** to find the best settings — the test set is never touched during tuning.

In [ ]:
# Pre-transform all 3 splits for XGBoost
preprocessor_xgb = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CAT_COLS)
])
X_train_xgb = preprocessor_xgb.fit_transform(X_train)   # fit ONLY on train!
X_val_xgb   = preprocessor_xgb.transform(X_val)
X_test_xgb  = preprocessor_xgb.transform(X_test)
print(f'Transformed shapes — Train: {X_train_xgb.shape} | Val: {X_val_xgb.shape} | Test: {X_test_xgb.shape}')

In [ ]:
# Optuna objective: tune on VALIDATION set (not test!)
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'random_state': 42, 'tree_method': 'hist', 'n_jobs': -1
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train_xgb, y_train, verbose=False)
    val_pred = model.predict(X_val_xgb)
    return np.sqrt(mean_squared_error(y_val, val_pred))   # minimise validation RMSE

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\n✅ Best Validation RMSE: {study.best_value:.4f}')
print('Best hyperparameters:', study.best_params)

In [ ]:
# Train final model with best params
best_xgb = xgb.XGBRegressor(
    **study.best_params,
    random_state=42, tree_method='hist', n_jobs=-1
)
best_xgb.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],   # monitor validation during training
    verbose=False
)

# Wrap in a simple object so evaluate_split works the same way
class XGBWrapper:
    def __init__(self, model, preprocessor):
        self.model = model
        self.pre   = preprocessor
    def predict(self, X):
        return self.model.predict(self.pre.transform(X))

xgb_wrapped = XGBWrapper(best_xgb, preprocessor_xgb)

all_results['XGBoost + Optuna (Best)'] = evaluate_split(
    'XGBoost + Optuna (Best)',
    xgb_wrapped, X_train, y_train, X_val, y_val, X_test, y_test
)

plot_predictions(y_test.values, xgb_wrapped.predict(X_test), 'XGBoost + Optuna')

In [ ]:
# Optuna optimization history
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optuna — Validation RMSE per Trial (lower = better)')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP explainability for XGBoost
cat_names_xgb = list(preprocessor_xgb.named_transformers_['cat'].get_feature_names_out(CAT_COLS))
all_feature_names = NUM_COLS + cat_names_xgb

explainer   = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test_xgb)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_xgb, feature_names=all_feature_names,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance — XGBoost\n(which features drive yield predictions most)')
plt.tight_layout(); plt.show()

shap.summary_plot(shap_values, X_test_xgb, feature_names=all_feature_names, show=False)
plt.title('SHAP Beeswarm — How each feature pushes predictions up or down')
plt.tight_layout(); plt.show()

---
## 7 — Full Model Comparison (All 3 Models × All 3 Splits)

In [ ]:
# Build a clean side-by-side comparison table
rows = []
for model_name, res_df in all_results.items():
    for split in ['Train', 'Validation', 'Test']:
        row = {'Model': model_name, 'Split': split}
        row.update(res_df.loc[split].to_dict())
        rows.append(row)

summary = pd.DataFrame(rows)
print('\n===== COMPLETE MODEL ACCURACY REPORT =====')
print(summary.to_string(index=False))
summary

In [ ]:
# Visual comparison — R² across models and splits
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
metrics_to_plot = [('R²', True), ('RMSE', False), ('MAE', False)]
colors = {'Train': '#3498db', 'Validation': '#f39c12', 'Test': '#2ecc71'}

for ax, (metric, higher_better) in zip(axes, metrics_to_plot):
    model_names_short = ['Ridge\n(Baseline)', 'Random\nForest', 'XGBoost\n+Optuna']
    x = np.arange(len(all_results))
    width = 0.25
    for i, split in enumerate(['Train', 'Validation', 'Test']):
        vals = [all_results[m].loc[split, metric] for m in all_results]
        bars = ax.bar(x + i*width, vals, width, label=split,
                      color=colors[split], edgecolor='black', alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
    ax.set_xticks(x + width)
    ax.set_xticklabels(model_names_short, fontsize=9)
    ax.set_title(f'{metric} ({"higher=better" if higher_better else "lower=better"})')
    ax.legend(fontsize=8)

plt.suptitle('Model Accuracy: Train / Validation / Test Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Quick summary of TEST SET results only (final honest scores)
print('\n====== FINAL TEST SET RESULTS (Honest Evaluation) ======')
test_only = summary[summary['Split'] == 'Test'][['Model','R²','MAE','RMSE','MAPE (%)']]
test_only = test_only.sort_values('R²', ascending=False).reset_index(drop=True)
test_only.index += 1
print(test_only.to_string())
print('\n💡 Higher R² = better  |  Lower MAE/RMSE/MAPE = better')
print(f'   Best model: {test_only.iloc[0]["Model"]}')

---
## 8 — Save Models

In [ ]:
import joblib, os
os.makedirs('saved_models', exist_ok=True)

joblib.dump(ridge_pipe,       'saved_models/ridge_baseline.pkl')
joblib.dump(rf_pipe,          'saved_models/random_forest.pkl')
joblib.dump(best_xgb,         'saved_models/xgboost_best.pkl')
joblib.dump(preprocessor_xgb, 'saved_models/xgboost_preprocessor.pkl')
print('✅ All models saved')

from google.colab import files
for f in ['ridge_baseline.pkl','random_forest.pkl','xgboost_best.pkl','xgboost_preprocessor.pkl']:
    files.download(f'saved_models/{f}')

---
## 9 — Inference: Predict on New Data

In [ ]:
# ← Replace these values with your real field measurements
new_sample = pd.DataFrame([{
    'seed_variety':      'Jet 999',   # Jet 999 | GT 709 | GT 200 | Pacific 808 | Commando
    'cob_height_cm':     110.0,
    'plant_height_cm':   220.0,
    'cob_wet_weight_g':  190.0,
    'cob_length_cm':     15.5,
    'num_seed_rows':     14,
}])

# Must add engineered features (same as training)
new_sample['cob_to_plant_ratio'] = new_sample['cob_height_cm'] / new_sample['plant_height_cm']
new_sample['weight_per_row']     = new_sample['cob_wet_weight_g'] / new_sample['num_seed_rows']

pred_ridge = ridge_pipe.predict(new_sample)[0]
pred_rf    = rf_pipe.predict(new_sample)[0]
pred_xgb   = xgb_wrapped.predict(new_sample)[0]

print('\n===== Predictions for New Field Sample =====')
print(f'  Ridge Regression  (Baseline) : {pred_ridge:.4f} Kg m⁻²')
print(f'  Random Forest     (Accurate) : {pred_rf:.4f} Kg m⁻²')
print(f'  XGBoost + Optuna  (Best)     : {pred_xgb:.4f} Kg m⁻²')
print(f'\n  ✅ Recommended prediction    : {pred_xgb:.4f} Kg m⁻²')